# 4장 — 신경망 기초: 세지 않고 배운다 (실습)

교재 `docs/book/04-neural-network.md` 와 함께 본다. 이 노트북에서 하는 것:

1. 숫자 하나짜리 자동미분 `Value` 로 역전파가 정확히 무엇을 하는지 본다 (PyTorch 와 대조)
2. NumPy 로 순전파·역전파를 손으로 써서 1장의 바이그램 확률표를 **학습으로** 얻는다
3. PyTorch 로 MLP 언어모델(Bengio 2003)을 학습하고 1장 n-gram 과 비교한다

> 전체 실행 약 4분. §2 의 NumPy 학습 ~1분, §3 의 MLP 학습 두 번 ~2분. (`setup_cpu()` 가 스레드를 물리 코어 수로 맞춘다 — 16 스레드로 돌리면 16배 느려진다. 7장.)

## 1. 자동미분 — 연쇄법칙을 코드로

In [ ]:
import math
import time

import numpy as np
import torch

from shllm.autograd import Value
from shllm.config import TOKENIZER_DIR, setup_cpu

setup_cpu()

x = Value(0.4)
y = Value(0.6)
z = (x * y + x).tanh() * 4  # z = 4·tanh(0.4·0.6 + 0.4)
z.backward()
print("z =", round(z.data, 4), "| dz/dx =", round(x.grad, 4), "| dz/dy =", round(y.grad, 4))

# 같은 식을 PyTorch 로
tx = torch.tensor(0.4, requires_grad=True)
ty = torch.tensor(0.6, requires_grad=True)
tz = torch.tanh(tx * ty + tx) * 4
tz.backward()
print("torch: dz/dx =", round(tx.grad.item(), 4), "| dz/dy =", round(ty.grad.item(), 4))

`Value` 는 연산할 때마다 "무엇으로부터 어떻게 만들어졌나"를 기억하고, `backward()` 가 출력에서 입력 쪽으로 거꾸로 가며 국소 미분을 곱해 나간다(연쇄법칙).
`torch.Tensor` 의 `requires_grad=True` · `.backward()` · `.grad` 가 정확히 같은 일을 텐서 단위로 한다. `src/shllm/autograd.py` 는 100줄이 안 된다 — 읽어 보라.

**기울기(gradient)의 뜻**: `x.grad = 4·(1−tanh²)·(y+1)` 은 "x 를 아주 조금 늘리면 z 가 그 몇 배로 변하는가". 학습은 loss 의 기울기를 구해 **반대 방향으로** 파라미터를 조금 옮기는 것의 반복이다.

## 2. 손으로 쓴 신경망 — 바이그램을 학습으로

모델: `logits = W[x]` (W 는 V×V), `p = softmax(logits)`, `loss = -mean(log p[정답])`.
1장에서는 W 에 해당하는 표를 **세어서** 만들었다. 이번엔 W 를 0 에서 출발시키고 loss 의 기울기를 따라 내려간다.

In [ ]:
from shllm.bigram import BigramModel
from shllm.data import load_corpus
from shllm.numpy_lm import backward, forward, sgd_step
from shllm.tokenizer import CharTokenizer

text = load_corpus("korean-classics")
ctok = CharTokenizer.from_text(text)
V = ctok.vocab_size
ids = np.array(ctok.encode(text))
n = int(0.9 * len(ids))
train_np, val_np = ids[:n], ids[n:]

W = np.zeros((V, V), dtype=np.float32)  # 0 → 모든 행이 균등분포 = loss log V
rng = np.random.default_rng(0)
losses = []
t0 = time.perf_counter()
for step in range(300):
    s = rng.integers(0, len(train_np) - 1, 4096)  # 무작위 위치 4,096곳 = 미니배치
    x, y = train_np[s], train_np[s + 1]
    loss, p = forward(W, x, y)
    W = sgd_step(W, backward(p, x, y, V), lr=300.0)
    losses.append(loss)
    if step % 50 == 0:
        print(f"step {step:>4}  loss {loss:.3f}")
print(f"{time.perf_counter() - t0:.0f}초")
vs = val_np[:20000]  # 전체 val 은 softmax 행렬 (11만 × 2,827) 이 1.3GB 라 앞 2만 토큰만
print(f"학습 후 val loss {forward(W, vs[:-1], vs[1:])[0]:.3f}   |  1장 세어서 만든 표: {BigramModel(V).fit(torch.tensor(train_np)).loss(torch.tensor(vs)):.3f}   |  균등 log V = {math.log(V):.3f}")

0 에서 출발한 W 가 300 스텝 만에 1장의 표 쪽으로 내려간다(아직 덜 수렴했지만 더 돌리면 계속 가까워진다). **세는 것과 배우는 것이 같은 곳에 도착한다** — 차이는 배우는 쪽이 "표"가 아닌 어떤 함수든 같은 방법으로 다룰 수 있다는 것이다.

역전파는 딱 세 줄이었다(`numpy_lm.backward`): `dlogits = p − onehot(y)` → `/ N` → `dW[x] += dlogits`. softmax 와 cross-entropy 를 합치면 미분이 "예측 확률 − 정답" 이 되는 것이 딥러닝에서 가장 자주 쓰이는 사실이다.

### 2.1 학습률

In [ ]:
for lr in (3.0, 30.0, 300.0, 1000.0):
    W = np.zeros((V, V), dtype=np.float32)
    rng = np.random.default_rng(0)
    with np.errstate(divide="ignore", over="ignore"):  # 발산하면 log 0 = -inf 경고가 나오므로 조용히
        for step in range(60):
            s = rng.integers(0, len(train_np) - 1, 4096)
            loss, p = forward(W, train_np[s], train_np[s + 1])
            W = sgd_step(W, backward(p, train_np[s], train_np[s + 1], V), lr=lr)
    print(f"lr={lr:>6}: 60 스텝 후 loss {loss:.3f}")

학습률이 너무 작으면 느리고, 너무 크면 발산한다(`inf`). 7장의 학습 루프는 처음엔 작게 시작해(warmup) 점점 줄이는 스케줄을 쓴다.

## 3. PyTorch 로 — MLP 언어모델

이제 표 대신 **함수**를 배운다. Bengio (2003): 직전 T 개 토큰을 각각 벡터로 바꿔(3장) 이어 붙이고, 은닉층 하나(tanh)를 거쳐 어휘 점수를 낸다.
1장 n-gram 과 결정적 차이 — 문맥이 dict 의 키가 아니라 **벡터**라서, 비슷한 문맥끼리 정보를 공유한다.

In [ ]:
from shllm.data import get_batch
from shllm.mlp import MLPLanguageModel, train_steps
from shllm.tokenizer import BPETokenizer

tok = BPETokenizer.load(TOKENIZER_DIR / "bpe-8192.json")
data = torch.tensor(tok.encode(text))
n = int(0.9 * len(data))
train, val = data[:n], data[n:]
Vb = tok.vocab_size

x, y = get_batch(train, block_size=4, batch_size=2)
print("x", x.tolist())
print("y", y.tolist(), "← x 를 한 칸 민 것. MLP 는 y 의 마지막 열만 쓴다")

In [ ]:
def run_mlp(block_size: int, steps: int = 3000, seed: int = 0):
    torch.manual_seed(seed)
    g = torch.Generator().manual_seed(seed)
    model = MLPLanguageModel(Vb, block_size=block_size, n_embd=64, n_hidden=256)

    def batch():
        bx, by = get_batch(train, block_size, 128, g)
        return bx, by[:, -1]

    t0 = time.perf_counter()
    losses = train_steps(model, batch, steps=steps, lr=3e-3, log_every=1000)
    with torch.no_grad():
        vx, vy = get_batch(val, block_size, 4096, torch.Generator().manual_seed(1))
        _, val_loss = model(vx, vy[:, -1])
    n_params = sum(p.numel() for p in model.parameters())
    print(f"T={block_size}: 파라미터 {n_params:,}  {time.perf_counter() - t0:.0f}초  train {np.mean(losses[-100:]):.3f}  val {val_loss.item():.3f}")
    return model, losses, val_loss.item()


mlp1, losses1, val1 = run_mlp(block_size=1)
mlp4, losses4, val4 = run_mlp(block_size=4)

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager

cjk = [f.name for f in font_manager.fontManager.ttflist if "CJK" in f.name]
if cjk:
    matplotlib.rcParams["font.family"] = cjk[0]


def smooth(v, k=50):
    return np.convolve(v, np.ones(k) / k, mode="valid")


fig, ax = plt.subplots(figsize=(6.5, 3.5))
ax.plot(smooth(losses1), label="MLP T=1 (바이그램)")
ax.plot(smooth(losses4), label="MLP T=4")
ax.set_xlabel("스텝")
ax.set_ylabel("train loss (이동평균)")
ax.legend()
plt.show()

### 3.1 1장 n-gram 과 비교 — 같은 토큰(BPE 8192), 같은 val

In [ ]:
from shllm.bigram import NGramModel

print(f"{'모델':<14}{'val loss':>9}")
for k in (2, 3, 5):
    print(f"{k}-gram (세기){NGramModel(k, Vb).fit(train).loss(val):>9.3f}")
print(f"{'MLP T=1':<14}{val1:>9.3f}")
print(f"{'MLP T=4':<14}{val4:>9.3f}")

읽는 법 (수치는 셀 출력):

- **MLP T=1 (신경 바이그램) 이 세어서 만든 2-gram 보다 낫다.** 같은 정보(직전 토큰 하나)인데 학습 쪽이 이긴다 — 임베딩(3장)으로 비슷한 토큰끼리 통계를 공유하기 때문이다. 세기는 토큰마다 따로 센다.
- **MLP T=4 는 5-gram 보다 훨씬 낫지만, T=1 보다는 아직 나쁘다.** train loss 는 T=4 가 더 낮고 val 은 더 높다 — 1장에서 본 과적합이다. 3,000 스텝 × 128 = 38만 토큰, 코퍼스 한 바퀴도 못 돌았다. 문맥이 길수록 파라미터가 데이터를 외우기 쉬워 더 긴 학습·드롭아웃·weight decay 가 필요하다 (7장).
- n-gram 은 문맥이 길어질수록 표가 비어 val 이 **나빠졌지만**(1장), MLP 는 문맥을 벡터로 다루기 때문에 본 적 없는 문맥 조합에도 **일반화**한다. 파라미터 270만 개는 5-gram 이 저장하던 41만 개 문맥 표와 비슷한 크기다.

### 3.2 생성

In [ ]:
prompt = tok.encode("옛날 옛적에 호랑이가")
g = torch.Generator().manual_seed(1337)
out = mlp4.generate(torch.tensor([prompt]), max_new_tokens=60, generator=g)
print(tok.decode(out[0].tolist()))

아직 문장이 되진 않지만 1장 바이그램보다 어절이 산다. 한계도 분명하다 — 문맥 길이 T 가 **고정**이고, 자리마다 가중치가 따로라 "3칸 앞의 토큰" 과 "4칸 앞의 토큰" 이 같은 단어여도 다르게 취급한다. 문맥을 늘리려면 `hidden` 층의 입력이 T·C 로 선형 증가한다.
5장의 어텐션은 이 문제를 푼다: 문맥 길이가 가변이고, 위치가 아니라 **내용**으로 어디를 볼지 정한다.

## 정리

- 자동미분 = 연산 그래프를 거꾸로 따라가며 국소 미분을 곱하기. `Value` 100줄 = `torch.autograd` 의 원리.
- 학습 = loss 의 기울기 반대 방향으로 파라미터를 조금씩 옮기기. 세어서 만든 표에 같은 loss 로 도착한다.
- `softmax + cross_entropy` 의 미분은 `p − onehot(y)`.
- MLP 언어모델: 문맥을 벡터로 → n-gram 이 못 하던 일반화. 그러나 문맥 길이 고정·위치별 가중치.

---
**다음 장**: 5장 — 셀프 어텐션. "어디를 볼지"를 내용으로 정한다.